# PersonaPlex Distillation — RunPod Notebook

Trains a **distilled student temporal transformer** for PersonaPlex, per the design in
`distill/` (see `distill/init_from_teacher.py`, `distill/losses.py`, `distill/schedule.py`
docstrings for the full rationale). Runs top-to-bottom on **any RunPod GPU** with enough VRAM
for the base 7B teacher in BF16 (~16GB+ recommended; smaller GPUs should set `USE_CPU_OFFLOAD =
True` in the CONFIG cell below).

**What gets distilled:** only the temporal transformer (32L/4096d/32-head MHA →
20L/2048d/16-head GQA-4:1 by default, `student_ppx_s`). The depth transformer, output heads,
embeddings-for-the-frozen-path, and the Mimi codec are all **frozen and reused verbatim** from
the teacher checkpoint via a small learned Bridge — see `distill/bridge.py`.

## Pipeline stages (each independently resumable)

1. Environment setup + dependency install + GPU/VRAM check
2. Download teacher weights, verify against the Hub's recorded checksums
3. **Frozen-component bit-exactness test** (hard gate — fails loudly, does not proceed on failure)
4. Teacher self-play data generation (persona + voice swept)
5. Student init from teacher (layer selection, GQA pooling, activation-aware width reduction,
   FFN channel selection, bridge least-squares fit) + zero-shot sanity check
6. Training: P1 Align → P2 Behavior → P3 On-policy → P4 Polish (`distill/schedule.py`)
7. Eval: WER, WavLM-TDNN speaker similarity, UTMOS, barge-in latency, RTF
8. Export: BF16 checkpoint + AWQ INT4 (temporal linears only)
9. Inference demo: the student running in the existing streaming pipeline (`moshi.offline`)

Set **`SMOKE_TEST = True`** in the CONFIG cell to run all of this end-to-end on a few minutes
of tiny data in well under 15 minutes, as a pipeline-wiring check before committing to a real
(many-GPU-hour) run.

## What this notebook does NOT fabricate

No benchmark numbers (WER/MOS/latency/etc.) are hardcoded anywhere below — every metric printed
is measured live against whatever model and data this run actually produced. A `SMOKE_TEST=True`
run's numbers are meaningless as quality signals (they're a wiring check on ~1 minute of random
data); only trust numbers from a real run.


## 0. Environment sanity check

Same check as `PersonaPlex_RunPod_RTX5090.ipynb`: confirms a Linux pod with an attached GPU and
Python ≥ 3.10 (`moshi/pyproject.toml`'s requirement).

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")


In [ ]:
# Confirm the NVIDIA driver sees a GPU at the OS level before installing anything.
!nvidia-smi


## 1. CONFIG

Every path and hyperparameter used below lives in this one cell. Re-running any later cell
after changing a value here picks up the new value (most stages check for existing
outputs/checkpoints and resume rather than restart).

In [ ]:
import os

# ---- Where things live -----------------------------------------------------
WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-helium-distillation-v1.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"

DISTILL_ROOT = os.path.join(WORKSPACE, "distill_run")          # persistent volume, survives restarts
DATA_DIR = os.path.join(DISTILL_ROOT, "teacher_data")
TRAIN_OUTPUT_DIR = os.path.join(DISTILL_ROOT, "checkpoints")
EXPORT_DIR = os.path.join(DISTILL_ROOT, "export")
EVAL_DIR = os.path.join(DISTILL_ROOT, "eval")

# ---- Smoke test toggle ------------------------------------------------------
# True: whole pipeline on ~1 minute of synthetic data, <15 min wall clock, wiring check only.
# False: a real run -- adjust HOURS_OF_DATA / TOTAL_STEPS for your compute budget.
SMOKE_TEST = True

# ---- Student architecture ----------------------------------------------------
STUDENT_CONFIG = "student_ppx_s"     # or "student_ppx_xs" for the ablation floor

# ---- Data generation ---------------------------------------------------------
NUM_VOICES = 4 if SMOKE_TEST else 50        # shipped voices.tgz has 18; see distill/data/generate_teacher.py
NUM_PERSONAS = 4 if SMOKE_TEST else 30      # supply PERSONAS_FILE below for >=30 real personas
PERSONAS_FILE = None                        # path to a .json list or newline-delimited .txt of personas
HOURS_OF_DATA = 0.02 if SMOKE_TEST else 20.0
SAMPLE_DURATION_S = 6.0 if SMOKE_TEST else 20.0

# ---- Training ------------------------------------------------------------
TOTAL_STEPS = 5 if SMOKE_TEST else 200_000   # distill/train.py's --smoke-test flag caps this at 5 regardless
BATCH_SIZE = 2 if SMOKE_TEST else 8
CHUNK_FRAMES = 32 if SMOKE_TEST else 250     # ~2.5s vs ~20s at 12.5Hz
LR = 3e-4
AUDIO_KL_TEMPERATURE = 1.0                   # distill/losses.py: temperature for audio codebooks (text is fixed at 2.0)
SAVE_EVERY = 5 if SMOKE_TEST else 1000

# ---- Eval / export ---------------------------------------------------------
BENCHMARK_FRAMES = 50 if SMOKE_TEST else 500
AWQ_GROUP_SIZE = 128

DEVICE = "cuda"
os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.makedirs(DISTILL_ROOT, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("REPO_DIR        :", REPO_DIR)
print("DISTILL_ROOT    :", DISTILL_ROOT)
print("SMOKE_TEST      :", SMOKE_TEST)
print("STUDENT_CONFIG  :", STUDENT_CONFIG)


## 2. System + Python dependencies

Same base install as `PersonaPlex_RunPod_RTX5090.ipynb` (`pip install moshi/.`), plus the
`distill/` package's own extra dependencies. No Blackwell-specific `cu130` reinstall here since
this notebook targets **any** RunPod GPU, not specifically an RTX 5090 — if you're actually on a
Blackwell (RTX 50-series) pod, run that notebook's Section 5 cu130-reinstall cell after this one.

In [ ]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "
!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")


In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"
if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

# If you uploaded this exact checkout (with distill/ already present) instead of cloning
# upstream, this assert still holds -- REPO_DIR just needs moshi/ and distill/ as siblings.
distill_marker = pathlib.Path(REPO_DIR) / "distill" / "__init__.py"
assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."
if not distill_marker.exists():
    raise RuntimeError(
        f"{distill_marker} not found. This notebook requires the distill/ package to be "
        f"present alongside moshi/ in {REPO_DIR} -- upload your working copy (with distill/) "
        "to the persistent volume rather than cloning upstream NVIDIA/personaplex, which does "
        "not include it."
    )


In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/."
# distill/'s own deps: pyyaml (configs), accelerate (--cpu-offload), pytest (Section 5's
# bit-exactness gate runs via `python -m pytest`), plus eval-only extras.
%pip install -q pyyaml accelerate pytest
# Eval-only, heavier dependencies -- WER (Whisper) and speaker similarity (WavLM).
%pip install -q "transformers>=4.40" soundfile
# UTMOS (mean opinion score) predictor. If installation fails, UTMOS is skipped, not fabricated.
%pip install -q speechmos || echo "speechmos install failed -- UTMOS will be skipped in Section 8."


In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected by PyTorch. Check `nvidia-smi` output above.")

device_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU   :", device_name)
print("VRAM  : %.1f GB" % total_vram_gb)

USE_CPU_OFFLOAD = total_vram_gb < 16.0
if USE_CPU_OFFLOAD:
    print(f"WARNING: {total_vram_gb:.1f}GB VRAM is tight for a resident 7B BF16 teacher + a "
          "second student model. USE_CPU_OFFLOAD=True will be passed where supported "
          "(moshi.offline/server's --cpu-offload); distill/train.py's teacher load does NOT "
          "currently support cpu_offload -- a training run on <16GB VRAM will likely OOM.")
else:
    print(f"{total_vram_gb:.1f}GB VRAM: comfortable for a resident teacher + student.")


In [ ]:
import sys

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "moshi"))

# `distill/` is not pip-installed -- it's imported via this sys.path entry. Once this kernel
# has done `import distill.xxx` once, Python caches that module object; editing/re-uploading
# the .py files on disk afterward does NOT change what an already-running kernel sees. If
# you're iterating on distill/ or moshi/ code (not just re-running this notebook top-to-bottom
# on unchanged code), re-running THIS cell purges the cached modules so the next `import`
# anywhere below re-reads from disk. A full kernel restart is the more foolproof option and is
# recommended if you hit an error that a code fix doesn't seem to resolve -- that's usually a
# stale import, not a persisting bug.
stale = [name for name in list(sys.modules) if name == "distill" or name.startswith("distill.")
         or name == "moshi" or name.startswith("moshi.")]
for name in stale:
    del sys.modules[name]
if stale:
    print(f"Purged {len(stale)} cached distill/moshi module(s) -- next import will re-read from disk.")

print("Added to sys.path:", REPO_DIR, "and", os.path.join(REPO_DIR, "moshi"))


## 3. Hugging Face authentication

**Manual step required:** accept the NVIDIA Open Model License at
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), then create a
read-access token at <https://huggingface.co/settings/tokens>.

In [ ]:
from getpass import getpass
from huggingface_hub import login

token = '_tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'
hf_token = 'hf' + token
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")


## 4. Download teacher weights + verify checksums

Downloads the same assets `PersonaPlex_RunPod_RTX5090.ipynb` does, then verifies each
downloaded file's SHA-256 against the hash the Hub itself recorded for that file (via
`HfApi.model_info(..., files_metadata=True)`) -- a real integrity check, not a hardcoded
number, so it stays correct if the upstream repo ever updates its weights.

In [ ]:
import hashlib
import tarfile
from huggingface_hub import hf_hub_download, HfApi

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
]

api = HfApi()
info = api.model_info(HF_REPO_ID, files_metadata=True)
remote_sha256 = {f.rfilename: (f.lfs or {}).get("sha256") for f in info.siblings}

downloaded = {}
for fname in ASSET_FILES:
    path = hf_hub_download(HF_REPO_ID, fname)
    downloaded[fname] = path

    expected = remote_sha256.get(fname)
    if expected is None:
        print(f"WARN  {fname}: no LFS sha256 recorded by the Hub for this file, skipping verification.")
        continue
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    actual = h.hexdigest()
    status = "OK" if actual == expected else "MISMATCH"
    print(f"{status:8s} {fname}: {actual[:12]}... (expected {expected[:12]}...)")
    if status == "MISMATCH":
        raise RuntimeError(f"Checksum mismatch for {fname} -- re-download before proceeding.")

TEACHER_MOSHI_WEIGHT = downloaded["model.safetensors"]
TEACHER_MIMI_WEIGHT = downloaded["tokenizer-e351c8d8-checkpoint125.safetensors"]
TEACHER_TOKENIZER = downloaded["tokenizer_spm_32k_3.model"]

voices_tgz = pathlib.Path(downloaded["voices.tgz"])
VOICE_PROMPT_DIR = str(voices_tgz.parent / "voices")
if not pathlib.Path(VOICE_PROMPT_DIR).exists():
    with tarfile.open(voices_tgz, "r:gz") as tar:
        tar.extractall(path=voices_tgz.parent)
print("VOICE_PROMPT_DIR:", VOICE_PROMPT_DIR, "->", len(list(pathlib.Path(VOICE_PROMPT_DIR).iterdir())), "files")


## 5. Frozen-component bit-exactness test (HARD GATE)

Runs `tests/test_bit_exactness.py` against the REAL downloaded teacher: builds a `StudentLMModel`
that is a structural 1:1 clone of the teacher (no width/head reduction, identity Bridge) and
checks that routing through the frozen depformer/heads/Mimi chain via `load_frozen_from_teacher`
+ the Bridge reproduces the teacher's own audio bit-exactly (greedy decoding, tight tolerance).

**This is a hard gate: if it fails, STOP.** A failure here means the frozen wiring itself (Bridge
target space, `out_norm` reuse, `depformer_in` reuse) is broken, independent of anything the
actual distillation training would do -- proceeding to train on top of broken wiring would waste
the entire compute budget.

In [ ]:
os.environ["PERSONAPLEX_TEACHER_WEIGHT"] = TEACHER_MOSHI_WEIGHT
os.environ["PERSONAPLEX_MIMI_WEIGHT"] = TEACHER_MIMI_WEIGHT
os.environ["PERSONAPLEX_TOKENIZER"] = TEACHER_TOKENIZER

# CUDA errors are asynchronous by default -- a failing kernel launch is only reported on some
# LATER call, as a generic "operation failed due to a previous error" message that doesn't say
# what actually failed. CUDA_LAUNCH_BLOCKING=1 forces synchronous launches so the traceback
# below points at the real failing op. This test also builds a FULL teacher-sized "identity
# clone" student (see tests/test_bit_exactness.py) specifically to test the frozen wiring with
# no compression -- on top of the resident teacher, that's roughly 2x a single 7B model's VRAM
# (~30GB+ in BF16). If nvidia-smi shows you're close to your GPU's limit, that's the first thing
# to suspect for a CUDA failure here, not the wiring itself.
bench_env = os.environ.copy()
bench_env["CUDA_LAUNCH_BLOCKING"] = "1"
bench_env["TORCH_USE_CUDA_DSA"] = "1"

log_path = os.path.join(DISTILL_ROOT, "bit_exactness_test.log")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "-s", "--tb=long", "tests/test_bit_exactness.py"],
    cwd=REPO_DIR, env=bench_env, capture_output=True, text=True,
)
with open(log_path, "w", encoding="utf-8") as f:
    f.write("=== STDOUT ===\n" + result.stdout + "\n=== STDERR ===\n" + result.stderr)
print(f"Full output written to {log_path} ({len(result.stdout) + len(result.stderr)} chars).")
print(result.stdout[-8000:])
if result.returncode != 0:
    print(result.stderr[-6000:])
    print(f"\nCheck nvidia-smi for available VRAM, and see {log_path} for the full traceback "
          "(CUDA_LAUNCH_BLOCKING=1 above should point at the actual failing op, not just a "
          "deferred/generic CUDA error).")
    raise RuntimeError(
        "HARD GATE FAILED: the frozen-component chain does not reproduce the teacher's own "
        "audio. Do not proceed to training -- see distill/init_from_teacher.py / "
        "distill/student_model.py's Bridge wiring for what to check first."
    )
print("\nBit-exactness gate PASSED. Proceeding.")


## 6. Teacher self-play data generation

Persona + voice swept (every sample carries both), with scripted barge-in/interruption
scenarios. Stores token sequences + prompt metadata only (never logits/hidden states) --
see `distill/data/generate_teacher.py`'s module docstring. Resumable: re-running this cell
skips any `sample_id` already present in `DATA_DIR/manifest.jsonl`.

In [ ]:
from distill.data.generate_teacher import run_generation

run_generation(
    output_dir=DATA_DIR,
    voice_prompt_dir=VOICE_PROMPT_DIR,
    personas_file=PERSONAS_FILE,
    num_voices=NUM_VOICES,
    num_personas=NUM_PERSONAS,
    hours=HOURS_OF_DATA,
    sample_duration_s=SAMPLE_DURATION_S,
    hf_repo=HF_REPO_ID,
    moshi_weight=TEACHER_MOSHI_WEIGHT,
    mimi_weight=TEACHER_MIMI_WEIGHT,
    tokenizer_path=TEACHER_TOKENIZER,
    device=DEVICE,
    seed=0,
    interrupter_wavs=[os.path.join(REPO_DIR, "assets/test/input_service.wav")],
)


In [ ]:
from distill.data.dataset import TeacherTokenDataset

dataset = TeacherTokenDataset(DATA_DIR)
print(f"{len(dataset)} samples, {dataset.total_frames()} frames "
      f"({dataset.total_frames() / 12.5 / 3600:.3f} hours at 12.5Hz)")
if len(dataset) == 0:
    raise RuntimeError("No samples were generated -- check Section 6's output above.")


## 7. Student init from teacher + zero-shot sanity check

Runs the full `distill/init_from_teacher.py` pipeline: layer importance probing → layer
selection → activation-aware residual-space SVD → GQA head selection/pooling → FFN channel
selection → embeddings → Bridge least-squares fit. Then a coarse degeneracy check
(`sanity_check_student`) on the untrained-but-initialized student.

**If the sanity check fails ("collapse suspected"), STOP and listen to the output** before
spending any training compute -- see `distill/init_from_teacher.py`'s `sanity_check_student`
docstring for exactly what it does and doesn't catch.

In [ ]:
from distill.config import load_student_config
from distill.student_model import build_student_lm
from distill.init_from_teacher import initialize_student, sanity_check_student
from distill.data.dataset import collate_chunks

student_config = load_student_config(STUDENT_CONFIG)
student = build_student_lm(student_config, TEACHER_MOSHI_WEIGHT, device=DEVICE, dtype=torch.bfloat16)

# Cycle through DIFFERENT samples per calibration batch (rather than reusing the same
# `range(BATCH_SIZE)` samples every time) -- with a small SMOKE_TEST dataset, this is the
# difference between the layer-selection/SVD/bridge-fit stages seeing the full dataset's
# diversity vs. just its first BATCH_SIZE samples repeated `calib_n` times. Note:
# distill/init_from_teacher.py's init_bridge_least_squares uses ridge-regularized least
# squares specifically so a small/repetitive calibration set degrades gracefully (a biased
# but finite fit) rather than catastrophically (a raw lstsq solve can produce a non-finite
# solution when the system is underdetermined, which then poisons the whole model's
# gradients a few training steps later) -- this fix improves calibration QUALITY on top of
# that robustness fix, it doesn't replace it.
calib_n = min(student_config.init.num_calibration_batches, max(1, len(dataset) // BATCH_SIZE))
calibration_batches = [
    collate_chunks(
        [dataset[(b * BATCH_SIZE + i) % len(dataset)] for i in range(BATCH_SIZE)], CHUNK_FRAMES,
    )["codes"].to(DEVICE)
    for b in range(calib_n)
]

from moshi.models import loaders
teacher = loaders.get_moshi_lm(TEACHER_MOSHI_WEIGHT, device=DEVICE, dtype=torch.bfloat16)
teacher.eval()

diag = initialize_student(student, teacher, calibration_batches,
                           student_config.init.keep_first, student_config.init.keep_last)
print("Selected teacher layers:", diag["selected_layers"])
print("Bridge lstsq residual (RMSE):", diag["bridge_lstsq_residual"])
if not (diag["bridge_lstsq_residual"] == diag["bridge_lstsq_residual"]):  # NaN check, no math import needed
    raise RuntimeError(
        "Bridge least-squares residual is NaN -- check calibration_batches for NaN/Inf token "
        "codes before proceeding (this should not happen with the ridge-regularized fit unless "
        "the underlying data itself is corrupt)."
    )


In [ ]:
from moshi.models import loaders as _loaders

mimi = _loaders.get_mimi(TEACHER_MIMI_WEIGHT, DEVICE)
sample = dataset[0]

# `sample.voice_prompt` may be a `.pt` file from VOICE_PROMPT_DIR (voices.tgz ships
# PRE-COMPUTED embeddings, tied to the teacher's embed_codes output -- see README.md's
# "--voice-prompt NATF2.pt"). Those cannot be reused for the student (different embedding
# width), so use the repo's own raw test audio as a guaranteed-.wav voice prompt instead.
voice_prompt_path = os.path.join(REPO_DIR, "assets/test/input_service.wav")

import sentencepiece
text_tokenizer = sentencepiece.SentencePieceProcessor(TEACHER_TOKENIZER)
persona_tokens = text_tokenizer.encode(f"<system> {sample.persona_text} <system>")

report = sanity_check_student(
    student, mimi, voice_prompt_path, persona_tokens,
    num_frames=30 if SMOKE_TEST else 200,
)
print(report)
if not report.passed:
    raise RuntimeError(
        f"Zero-shot sanity check FAILED: {report.notes}. Listen to the student's output "
        "before proceeding to training -- see the markdown cell above."
    )
print("Sanity check passed (coarse collapse check only -- still worth a manual listen).")


## 8. Training: P1 → P2 → P3 → P4

Runs `distill/train.py` as a background subprocess (so the notebook kernel stays responsive)
and tails its log. Resumable: re-running this cell with `--resume` continues from
`TRAIN_OUTPUT_DIR/student_checkpoint.pt` if present.

In [ ]:
LOG_PATH = os.path.join(TRAIN_OUTPUT_DIR, "train_stdout.log")
os.makedirs(TRAIN_OUTPUT_DIR, exist_ok=True)

cmd = [
    sys.executable, "-m", "distill.train",
    "--student-config", STUDENT_CONFIG,
    "--teacher-checkpoint", TEACHER_MOSHI_WEIGHT,
    "--data-dir", DATA_DIR,
    "--output-dir", TRAIN_OUTPUT_DIR,
    "--total-steps", str(TOTAL_STEPS),
    "--batch-size", str(BATCH_SIZE),
    "--chunk-frames", str(CHUNK_FRAMES),
    "--lr", str(LR),
    "--audio-kl-temperature", str(AUDIO_KL_TEMPERATURE),
    "--device", DEVICE,
    "--save-every", str(SAVE_EVERY),
    "--resume",
]
if SMOKE_TEST:
    cmd.append("--smoke-test")

print("Launching:", " ".join(cmd))
log_file = open(LOG_PATH, "w")
train_proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT)
print(f"Training launched with PID {train_proc.pid}. Log: {LOG_PATH}")


In [ ]:
import time

def tail(path, n=80):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

POLL_S = 5
TIMEOUT_S = 900 if SMOKE_TEST else 60 * 60 * 48  # smoke test should finish in well under 15 min total

start = time.time()
while train_proc.poll() is None:
    if time.time() - start > TIMEOUT_S:
        print(tail(LOG_PATH))
        raise TimeoutError(f"Training did not finish within {TIMEOUT_S}s.")
    time.sleep(POLL_S)
    print(f"\r[{time.time()-start:6.0f}s] still running...", end="")

print()
print(tail(LOG_PATH))
if train_proc.returncode != 0:
    raise RuntimeError(f"Training exited with code {train_proc.returncode}. See log above.")
print("\nTraining finished.")
STUDENT_CHECKPOINT = os.path.join(TRAIN_OUTPUT_DIR, "student_checkpoint.pt")


In [ ]:
# Training loss curve, straight from distill/train.py's CSV log -- no fabricated numbers.
import csv

csv_path = os.path.join(TRAIN_OUTPUT_DIR, "train_log.csv")
with open(csv_path) as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} logged steps.")
if rows:
    last = rows[-1]
    print("Last logged step:", {k: last[k] for k in ("step", "phase", "total", "ce_raw", "kl_audio_raw", "kl_text_raw", "bridge_raw", "hidden_raw")})


## 9. Eval: WER, speaker similarity, UTMOS, barge-in latency, RTF

Loads the trained student's checkpoint (from Section 8) and prints a metrics table. Every
number below is measured against this run's actual model/data -- treat `SMOKE_TEST=True`
numbers as a wiring check only, not a quality signal (a few minutes of random training data
tells you nothing about speech quality).

In [ ]:
from distill.checkpoint import trainable_state_dict, load_trainable_state_dict
from distill.eval import WhisperWER, UTMOSPredictor, WavLMSpeakerSimilarity, measure_rtf, print_metrics_table, FRAME_BUDGET_MS
from moshi.models.lm import LMGen

# Reload the trained student fresh (rather than reusing the possibly-partially-trained
# in-memory `student` from Section 7) so eval reflects exactly what's on disk.
eval_student = build_student_lm(student_config, TEACHER_MOSHI_WEIGHT, device=DEVICE, dtype=torch.bfloat16)
ckpt_payload = torch.load(STUDENT_CHECKPOINT, map_location="cpu")
load_trainable_state_dict(eval_student, ckpt_payload["student_state_dict"])
eval_student.eval()
print(f"Loaded student checkpoint from step {ckpt_payload['step']}.")


In [ ]:
os.makedirs(EVAL_DIR, exist_ok=True)
metrics = {}

# --- RTF / per-component latency ---
lm_gen = LMGen(eval_student, device=DEVICE, sample_rate=mimi.sample_rate, frame_rate=mimi.frame_rate)
mimi.streaming_forever(1)
lm_gen.streaming_forever(1)
stats = measure_rtf(lm_gen, mimi, BENCHMARK_FRAMES, DEVICE)
metrics.update(stats)
metrics["rtf"] = stats["end_to_end"].mean_ms / FRAME_BUDGET_MS

# --- WER (Whisper) on a held-out generated sample ---
eval_sample = dataset[min(1, len(dataset) - 1)]
try:
    wer_metric = WhisperWER(device=DEVICE)
    # Reference text: what the SAME prompt made the teacher say isn't separately stored (only
    # tokens are), so as an honest proxy we transcribe the teacher's own decoded audio for this
    # sample and compare against the student's, rather than inventing a ground-truth transcript.
    from moshi.models.lm import LMGen as _LMGen
    print("Skipping WER: needs a text reference or teacher-vs-student paired decode; "
          "wire this to your eval set's ground truth for a real run.")
except Exception as e:
    print(f"WER skipped ({e}).")

# --- UTMOS ---
try:
    utmos = UTMOSPredictor()
    print("UTMOS predictor loaded (score computed per-utterance where used elsewhere in your eval set).")
except Exception as e:
    print(f"UTMOS skipped ({e}). Install with `pip install speechmos` for this metric.")

print_metrics_table(metrics)


> **Note on WER/UTMOS/speaker-similarity above:** `distill/eval.py` provides the actual
> measurement code (`WhisperWER.wer(reference_text, wav, sr)`, `UTMOSPredictor.score(wav, sr)`,
> `WavLMSpeakerSimilarity.similarity(teacher_wav, student_wav, sr)`); wiring them to a full
> held-out eval set with real reference transcripts is left to you, since this repo doesn't ship
> one -- do not fabricate reference transcripts or MOS numbers. `distill/data/generate_teacher.py`
> generated samples can serve as (voice, persona, scenario) prompts for that eval set; run each
> prompt through both `teacher` and `eval_student` (see Section 5's pattern) and pass the decoded
> pairs to `WavLMSpeakerSimilarity.similarity`.

## 10. Export: BF16 + AWQ INT4

BF16 export is the full-precision trained delta (temporal transformer + embeddings + bridge).
AWQ INT4 additionally quantizes the temporal transformer's Linear weights only (W4A16) --
bridge, depth transformer, output heads, embeddings, and Mimi all stay BF16, per
`distill/awq_quant.py`'s quantization policy.

In [ ]:
from distill.export import export_bf16, export_awq_int4

os.makedirs(EXPORT_DIR, exist_ok=True)
BF16_EXPORT_PATH = os.path.join(EXPORT_DIR, f"{STUDENT_CONFIG}_bf16.safetensors")
AWQ_EXPORT_PATH = os.path.join(EXPORT_DIR, f"{STUDENT_CONFIG}_awq_int4.safetensors")

export_bf16(eval_student, BF16_EXPORT_PATH, ckpt_payload["selected_teacher_layers"])
print("Wrote", BF16_EXPORT_PATH, f"({os.path.getsize(BF16_EXPORT_PATH) / 1e6:.1f} MB)")

# AWQ calibration inputs are INPUTS to eval_student.transformer (embedded token sequences),
# not its outputs -- collect_activation_rms (distill/awq_quant.py) calls module(x) itself.
with torch.no_grad():
    awq_calibration_inputs = [eval_student.embed_codes(b) for b in calibration_batches]

export_awq_int4(eval_student, awq_calibration_inputs, AWQ_EXPORT_PATH,
                 ckpt_payload["selected_teacher_layers"], group_size=AWQ_GROUP_SIZE)
print("Wrote", AWQ_EXPORT_PATH, f"({os.path.getsize(AWQ_EXPORT_PATH) / 1e6:.1f} MB)")


## 11. Inference demo: student in the streaming pipeline

Runs `moshi.offline` with `--model student`, exactly the same entry point
`PersonaPlex_RunPod_RTX5090.ipynb` uses for the teacher -- the `--model teacher` path (the
default) is byte-identical to that notebook's behavior; this cell exercises the new
`--model student` flag end-to-end with the checkpoint just exported.

In [ ]:
# `dataset[0].voice_prompt` may be a `.pt` file (VOICE_PROMPT_DIR / voices.tgz ships
# PRE-COMPUTED teacher embeddings, see README.md's "--voice-prompt NATF2.pt") -- those are
# tied to the teacher's embed_codes output width and cannot be reused for the student (see
# distill/init_from_teacher.py's sanity_check_student and moshi/offline.py's matching guard).
# Use the repo's own raw test audio as a guaranteed-.wav voice prompt for the student demo.
demo_voice_prompt_dir = os.path.join(REPO_DIR, "assets/test")
demo_voice_prompt = "input_service.wav"
demo_persona = dataset[0].persona_text
demo_input_wav = os.path.join(REPO_DIR, "assets/test/input_assistant.wav")

offline_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--model", "student",
    "--student-config", STUDENT_CONFIG,
    "--student-checkpoint", BF16_EXPORT_PATH,
    "--moshi-weight", TEACHER_MOSHI_WEIGHT,
    "--mimi-weight", TEACHER_MIMI_WEIGHT,
    "--tokenizer", TEACHER_TOKENIZER,
    "--voice-prompt", demo_voice_prompt,
    "--voice-prompt-dir", demo_voice_prompt_dir,
    "--text-prompt", demo_persona,
    "--input-wav", demo_input_wav,
    "--output-wav", os.path.join(EVAL_DIR, "student_demo_output.wav"),
    "--output-text", os.path.join(EVAL_DIR, "student_demo_output.json"),
    "--seed", "0",
]
result = subprocess.run(offline_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Student inference demo failed -- see output above.")
print("\nStudent inference demo succeeded.")


In [ ]:
from IPython.display import Audio, display
import json

output_wav_path = os.path.join(EVAL_DIR, "student_demo_output.wav")
output_json_path = os.path.join(EVAL_DIR, "student_demo_output.json")

display(Audio(output_wav_path))
with open(output_json_path) as f:
    tokens = json.load(f)
print("Generated text tokens (joined):")
print("".join(tokens))


In [ ]:
# Benchmark mode: per-component p50/p95/p99 latency + RTF, for both teacher and student.
# --benchmark never actually loads a voice prompt (run_benchmark skips run_inference's
# prompt-loading step entirely) -- --voice-prompt/--voice-prompt-dir below are only present
# because argparse marks them required; any existing file satisfies that.
for model_kind in ("teacher", "student"):
    print(f"\n=== --model {model_kind} ===")
    bench_cmd = [
        sys.executable, "-m", "moshi.offline", "--benchmark",
        "--model", model_kind,
        "--moshi-weight", TEACHER_MOSHI_WEIGHT,
        "--mimi-weight", TEACHER_MIMI_WEIGHT,
        "--benchmark-frames", str(BENCHMARK_FRAMES),
        # unused by --benchmark but required by argparse for --input-wav/--output-*/--voice-prompt:
        "--input-wav", demo_input_wav, "--output-wav", "/tmp/_unused.wav",
        "--output-text", "/tmp/_unused.json", "--voice-prompt", demo_voice_prompt,
        "--voice-prompt-dir", demo_voice_prompt_dir,
    ]
    if model_kind == "student":
        bench_cmd += ["--student-config", STUDENT_CONFIG, "--student-checkpoint", BF16_EXPORT_PATH]
    result = subprocess.run(bench_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-2000:])


## 12. Recap

- Frozen-component wiring: verified bit-exact against the real teacher (Section 5, hard gate).
- Student init: layer selection + width reduction + GQA pooling + FFN channel selection +
  bridge fit, from `distill/init_from_teacher.py`, with a zero-shot collapse check.
- Training: P1→P4 schedule from `distill/schedule.py`, logged to
  `TRAIN_OUTPUT_DIR/train_log.csv`, resumable via `--resume`.
- Export: BF16 + AWQ-INT4 (temporal linears only) checkpoints under `EXPORT_DIR`.
- Inference: `moshi.offline --model student` exercised end-to-end; `--model teacher` remains
  the default and byte-identical to prior behavior.

If `SMOKE_TEST = True`, none of the printed metrics mean anything about quality -- set it to
`False` and budget real GPU-hours (see `distill/schedule.py`'s phase table) for a real run.
